# Profile the candidate list and the films

Investigation, not production code. It reads the id list as text and the films from the **cache**
(run `python run_pipeline.py` once first, so the cache exists), and it only reports: nothing here changes any data.

**To use it:** *Run All*. Each section ends in numbers you can quote in `docs/findings.md`.
**Before you commit:** clear all outputs, so no data ends up in version control.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "explore" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from pipeline import config
from pipeline.extract import fetch_all, read_id_list
from pipeline.transform import build_tables
from pipeline.validate import title_year_key, validate_ids

pd.set_option("display.max_colwidth", 80)
print("id list:", config.ID_FILE)
print("cache:  ", config.CACHE)

## 1. The candidate list, exactly as received

In [ ]:
lines = read_id_list()
ids, quarantine = validate_ids(lines)
print(f"{len(lines)} lines | {len(ids)} usable unique ids | {len(quarantine)} quarantined")
quarantine["reason"].value_counts()

In [ ]:
# The lines behind the numbers
quarantine[["source_line", "raw_value", "reason", "duplicate_of_line"]]

In [ ]:
# Values that arrived with spaces or leading zeros but were still usable
ids[ids["raw_value"].ne(ids["raw_value"].str.strip()) | ids["raw_value"].str.strip().str.startswith("0")]

## 2. The films, from the cache

In [ ]:
acq = fetch_all(ids["movie_id"].tolist(), offline=True)   # never calls the API
print("details:", acq["movie_status"].value_counts().to_dict())
films, cast, directors = build_tables(acq)
print(f"films {len(films)} | cast rows {len(cast)} | director rows {len(directors)}")

In [ ]:
def columns(df):
    '''One row per column: missing, distinct, zeros, and the range for numbers.'''
    num = df.select_dtypes("number")
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "distinct": df.nunique(),
        "zeros": (num == 0).sum().reindex(df.columns),
        "min": num.min().reindex(df.columns),
        "median": num.median().reindex(df.columns),
        "max": num.max().reindex(df.columns),
    })

columns(films)

## 3. The questions the brief makes us ask

One line per question, each answered with a number you can quote.

In [ ]:
released = films["status"].eq("Released")
checks = {
    "films fetched": len(films),
    "statuses": films["status"].value_counts().to_dict(),
    "budget = 0": int(films["budget"].eq(0).sum()),
    "revenue = 0": int(films["revenue"].eq(0).sum()),
    "budget or revenue above 0 but under $1,000": int((films["budget"].between(1, 999) | films["revenue"].between(1, 999)).sum()),
    "no release date": int(films["release_date"].isna().sum()),
    "released before 2000": int(films["release_year"].lt(2000).sum()),
    "in a collection": int(films["collection_id"].notna().sum()),
    "collections on the list": int(films["collection_id"].nunique()),
    "collections with one film on the list": int((films["collection_id"].value_counts() == 1).sum()),
    "same title + year under two ids": int(title_year_key(films).duplicated(keep=False).sum()),
    "runtime under 40 min": int(films["runtime"].between(1, 39).sum()),
    "films with no cast listed": int((~films["movie_id"].isin(cast["movie_id"])).sum()),
    "films with no director listed": int((~films["movie_id"].isin(directors["movie_id"])).sum()),
}
pd.Series(checks, dtype="object")

## 4. The rows behind the numbers

Counts tell you *how many*. Rows tell you *why*.

In [ ]:
# Revenue or budget of zero: gaps, or real?
films[films["revenue"].eq(0) | films["budget"].eq(0)][["movie_id", "title", "release_year", "status", "budget", "revenue", "collection_name"]]

In [ ]:
# Is what's missing different from what's present? (pre-2000 share, franchise share)
films.assign(pre_2000=films["release_year"].lt(2000), in_collection=films["collection_id"].notna()) \
     .groupby(films["revenue"].eq(0))[["pre_2000", "in_collection"]].mean().round(2)

In [ ]:
# Not released, or very short
films[~released | films["runtime"].between(1, 39)][["movie_id", "title", "status", "runtime", "release_date"]]

In [ ]:
# Same title and year under two ids
films[title_year_key(films).duplicated(keep=False)].sort_values("title")[["movie_id", "title", "release_year", "budget", "revenue"]]

In [ ]:
# Collections and how many of their films are on the list
films.groupby("collection_name")["movie_id"].count().sort_values(ascending=False)